# OpenMedicine — PDF Guideline to Markdown with Docling

This notebook converts clinical guideline PDFs into structured Markdown for ingestion into the GraphRAG knowledge graph.

**Important:** Use a **GPU runtime** for large guidelines (Runtime → Change runtime type → T4 GPU).

**Instructions:**
1. Set GPU runtime (recommended for documents > 50 pages)
2. Run all cells in order
3. Download the generated `.md` file from the output
4. Place it in your local project and run the ingestion CLI

In [ ]:
# Install docling (takes ~2-3 minutes on Colab)
!pip install -q docling

In [ ]:
import requests
from pathlib import Path

# === CONFIGURATION ===
# AHA/ACC/HFSA 2022 Heart Failure Guideline
# DOI: 10.1161/CIR.0000000000001063

PDF_URLS = [
    "https://www.ahajournals.org/doi/pdf/10.1161/CIR.0000000000001063",
    "https://achpccg.com/wp-content/uploads/2024/02/Heidenreich-et-al_2022_AHA-ACC-HFSA-Guideline-for-the-Management-of-Heart-Failure-A-Report-of-the-American-College-of-Cardiology-American-Heart-Association-Joint-Committee-on-Clinical-Practice-Guidelines.pdf",
]

GUIDELINE_ID = "aha_acc_hf_2022"
LOCAL_PDF = f"{GUIDELINE_ID}.pdf"
OUTPUT_FILE = f"{GUIDELINE_ID}.md"

# === DOWNLOAD PDF ===
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/pdf",
}

for url in PDF_URLS:
    print(f"Trying: {url[:80]}...")
    try:
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        Path(LOCAL_PDF).write_bytes(resp.content)
        print(f"Downloaded: {len(resp.content) / 1024 / 1024:.1f} MB")
        break
    except Exception as e:
        print(f"  Failed: {e}")
else:
    raise RuntimeError(
        "Could not download the PDF from any URL. "
        "Upload the PDF manually to this Colab session and set LOCAL_PDF to the filename."
    )

In [ ]:
from docling.document_converter import DocumentConverter

# Convert the full PDF — no page filtering
converter = DocumentConverter()
print(f"Converting {LOCAL_PDF} (this may take 10-30 minutes on CPU, ~2-5 min on GPU)...")
result = converter.convert(LOCAL_PDF)
doc = result.document
print(f"Conversion complete!")
print(f"Document name: {doc.name}")

In [ ]:
# Export as Markdown
markdown = doc.export_to_markdown()
print(f"Markdown length: {len(markdown):,} characters")
print(f"Estimated tokens: ~{len(markdown.split()):,}")
print()
print("First 3000 chars:")
print("=" * 80)
print(markdown[:3000])

In [ ]:
# Save and download
Path(OUTPUT_FILE).write_text(markdown, encoding="utf-8")
print(f"Saved to {OUTPUT_FILE}")

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("Download started!")
except ImportError:
    print(f"Not running in Colab. File saved locally at: {OUTPUT_FILE}")

## Next Steps

After downloading `aha_acc_hf_2022.md`, place it in your project and run:

```bash
# From the openmedicine project root:
cp ~/Downloads/aha_acc_hf_2022.md data/guidelines/

uv run python -m open_medicine.graphrag.ingest \
  --file data/guidelines/aha_acc_hf_2022.md \
  --id aha_acc_hf_2022 \
  --doi '10.1161/CIR.0000000000001063' \
  --title '2022 AHA/ACC/HFSA Guideline for the Management of Heart Failure' \
  --year 2022 \
  --org 'AHA/ACC/HFSA'
```